# step back prompt
以小台指期半小時為單位  
針對k線走勢評估接下來的漲跌

In [ ]:
# %pip install mplfinance

In [ ]:
import pandas as pd
import os

def read_taifex(path:str, product: str, product_m_or_w: str):
    '''
        讀取台灣期貨交易所的歷史資料，並選取特定商品代號及月份(週別)的資料

        年月日: str 
        商品代號: str
        商品月份: str

        回傳: DataFrame
    '''
    
    # 讀取 CSV 檔案，並將所有元素轉換成字串
    if os.path.exists(path):
        df = pd.read_csv(path, encoding="BIG5", dtype=str)
    else:
        return pd.DataFrame()
     
    # 移除所有元素的空白
    df = df.map(lambda x: x.strip())

    # 當 '到期月份(週別)' 包含 '/' 時，將 '成交價格' 的值改為 '近月價格' 的值
    df.loc[df['到期月份(週別)'].str.contains('/'), '成交價格'] = df['近月價格']
    
    # 到期月份(週別)中有些資料是202404/202405，代表轉倉，因此分割字串並選取第一個元素並存到新欄位 "商品月份"
    df['商品月份'] = df['到期月份(週別)'].str.split('/').str[0]

    df = df[df["商品代號"] == product]
    df = df[df["商品月份"] == product_m_or_w]

    # 將"交易日期" 和 "交易時間" 合併成一個新的欄位 "交易時間"
    df['Date'] = df['成交日期'] + df['成交時間']

    # 然後，將 "交易時間" 轉換成 datetime 對象
    # 假設 "交易日期" 是 "YYYYMMDD" 的格式，"交易時間" 是 "HHMMSS" 的格式
    df['Date'] = pd.to_datetime(df['Date'], format='%Y%m%d%H%M%S')

    df.set_index('Date', inplace=True)
    df = df.drop(columns=['成交日期'])
    df = df.drop(columns=['成交時間'])

    return df

In [ ]:
import pandas as pd
import os
from datetime import datetime, time
from dateutil.relativedelta import relativedelta
import matplotlib.pyplot as plt
import mplfinance as mpf

# 假設你的 DataFrame 是 df
def to_kline(interval:str, df:pd.DataFrame):
    '''
    將原始交易資料轉換為 K 線資料

    參數:
        interval: K 線的時間間隔，例如 '5min' 代表 5 分鐘
        df: 原始交易資料，必須包含 '成交數量(B+S)' 和 '成交價格' 兩個欄位

    回傳值
        df_kline: K 線資料，包含 'Open', 'High', 'Low', 'Close', 'Volume' 五個欄位
    '''

    df.index = pd.to_datetime(df.index)  # 確保索引是 datetime 對象
    df['成交數量(B+S)'] = df['成交數量(B+S)'].astype(int)
    df['成交價格'] = df['成交價格'].astype(float)

    # 計算 OHLC
    df_ohlc = df['成交價格'].resample(interval).agg({'Open':'first', 'High':'max', 'Low':'min', 'Close':'last'})

    # 計算 Volume
    df_volume = df['成交數量(B+S)'].resample(interval).sum()

    # 將兩個 DataFrame 合併
    df_kline = pd.concat([df_ohlc, df_volume], axis=1)
    df_kline = df_kline.rename(columns={'成交數量(B+S)': 'Volume'})  # 將 '成交數量(B+S)' 重新命名為 'Volume'

    return df_kline

def fetch_time(datetime:datetime, time_start:time=time(8,45), time_end:time=time(13,45), interval:str="5min", symbol:str="MTX"):
    year = datetime.strftime('%Y')
    month = datetime.strftime('%m')
    day = datetime.strftime('%d')

    path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), 'history_data', 'tw', 'taifex', f'Daily_{year}_{month}_{day}.csv')
    
    # 讀取檔案，如果當天的資料不存在，則讀取下一個月的資料(轉倉後的資料)
    df = read_taifex(path, symbol, year + month)
    if df.empty:
        df = read_taifex(path, symbol, year + (datetime + relativedelta(months=1)).strftime('%m'))
    if df.empty:
        return None
    df_kline = to_kline(interval, df)

    # 轉成固定時間區間
    # df_kline.index = pd.to_datetime(df_kline.index)
    iday = df_kline[df_kline.index.date == datetime.date()]
    iday = iday[iday.index.time >= time_start]
    iday = iday[iday.index.time < time_end]

    return iday

def draw_kline(iday:pd.DataFrame):
    '''
        繪製 K 線圖
        參數:
            iday : pd.DataFrame : 股票資料
        回傳:
            kline_path : str : 圖片檔案路徑
    '''

    mc = mpf.make_marketcolors(up='r',down='g',inherit=True)
    s  = mpf.make_mpf_style(base_mpf_style='tradingview',marketcolors=mc)
    if int(iday.shape[0]) == 0:
        return None, None
    
    # 繪製 K 線圖
    fig, axes = mpf.plot(iday.dropna(), type='candle', style=s,
            ylabel='price',
            volume=True,
            returnfig=True)
    plt.close()

    kline_path = os.path.join("img", "fig.png")
    fig.savefig(kline_path)

    return kline_path

In [ ]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain.schema.messages import HumanMessage
from langchain.schema.messages import SystemMessage
from langchain.schema.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

In [ ]:

# 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage 設定固定回答
# messages = [
#     SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
#     HumanMessage(content="請問k線對於市場情緒的概念是什麼?")
# ]

# # 生成第一次對話回應
# res1 = llm.invoke(messages)
# print(res1.content)
# with open('SBP_candidate/kline3.txt', 'w') as f:
#     f.write(res1.content)

In [ ]:
# k 線圖

from datetime import datetime, time
import matplotlib.pyplot as plt
import pandas as pd
import os
from datetime import timedelta
import json

# 創建日期範圍
start_date = "2024-01-01"
end_date = "2024-05-30"
date_range = pd.date_range(start=start_date, end=end_date)

start_time = datetime.strptime('09:00', '%H:%M')
end_time = datetime.strptime('13:00', '%H:%M')

time_range = []
while start_time <= end_time:
    time_range.append(start_time.time())
    start_time += timedelta(minutes=30)

# 讀取 k 
with open('SBP_candidate/kline1.txt', 'r', encoding='utf-8') as f:
    content = f.read()

# 對於每一天的每一小時調用 fetch_time
for date in date_range:
    for t1 in time_range:
        t2 = (datetime.combine(date, t1) + timedelta(minutes=30)).time()
        print(f"date: {date}, 從：{t1}～{t2}")
        
        df = fetch_time(date, t1, t2, "5min", "MTX")
        if df is None or len(df)<1:
            continue
        change = df['Close'].iloc[-1]/df['Open'].iloc[0]
        path = draw_kline(df)
        if path is None:
            continue

        # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
        messages = [
            SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
            HumanMessage(content="請問k線對於市場情緒的概念是什麼?")
        ]

        # 設定假的第一次對話回應 以減少使用次數
        res1 = AIMessage(content=content  , response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': [{'category': 'HARM_CATEGORY_SEXUALLY_EXPLICIT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HATE_SPEECH', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_HARASSMENT', 'probability': 'NEGLIGIBLE', 'blocked': False}, {'category': 'HARM_CATEGORY_DANGEROUS_CONTENT', 'probability': 'NEGLIGIBLE', 'blocked': False}]}, id='run-3bdfe58e-7707-4a1b-8f35-b8953592f5ff-0')

        # 將 AI 的回應添加到消息列表中
        messages.append(res1)
        messages.append(
            HumanMessage(
                content=[
                    {
                        "type": "text",
                        "text": "這是一張小台指的5分k線圖，請用上面的概念，並透過這張圖的具體形態，關鍵價格，成交量的變化趨勢，深入分析接下來半小時的市場情緒狀態，並猜測其隱含意義",
                    },
                    {"type": "image_url", "image_url": path}  # 替換為實際圖像路徑
                ]
            )
        )
        # print(messages)
        res2 = llm.invoke(messages)
        # # print("res2", res2.content)
        
        res3 = llm.invoke(f"請依據{res2.content}進行判斷，如果情緒好則只能回答 #long、如果不好則只能回答#short、如果不確定則只能回答 #unknown")
        sig_str = res3.content
        
        # sig_str = "long"
        sig = 0
        if sig_str.find("unknown") > 0:
            sig = 0
        elif sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        print(sig_str, sig)

        # 輸出部分 ==================================
        date_str = date.strftime("%Y%m%d")
        start_time_str = t1.strftime("%H_%M")
        # end_time_str = t2.strftime("%H")
        formatted_str = f"{date_str}_{start_time_str}"

        output_data = {
            "date": date_str,
            "start_time": start_time_str,
            "model": model,
            "temperature": temperature,
            "top_p": top_p,
            "seed": seed,
            "change": change,
            "response_sig": sig,
            "sig_str":sig_str,
        }
        json_content = json.dumps(output_data, ensure_ascii=False, indent=4)

        # print(formatted_str)
        out_path = os.path.join("out_kline_SBP_30min_1")
        os.makedirs(out_path, exist_ok=True)
        with open(out_path + "/" + formatted_str + ".json", 'w', encoding='utf-8') as f:
            f.write(json_content)
        


In [130]:
import pandas as pd
import json
import os
import random

# 計算準確率和平均報酬率
gain = []
loss = []

tp = 0
fp = 0
tn = 0
fn = 0

max_gain = 0
max_loss = 0

path = "out_kline_SBP/"
files = os.listdir(path)
# print(len(files))
files = sorted(files)
print((len(files)))

for idx, file in enumerate(files):
    if file.split("_")[1] == "13":
        continue
    # print(file)
    with open(path + file, "r", encoding='utf-8') as f:
        data = json.load(f)
        
        # sig = random.randint(-1, 1)
        sig_str = data["sig_str"]

        sig = 0
        if sig_str.find("unknown") > 0:
            sig = 0
        elif sig_str.find("long") > 0:
            sig = 1
        elif sig_str.find("short")>0:
            sig = -1
        else:
            sig = 0
        # print(sig_str, sig)
        # sig = data["response_sig"]

        # print(data["response_sig"], data["sig_str"])

    if idx+1 == len(files):
        break
    # print(files[idx+1])
    with open(path + files[idx+1], "r", encoding='utf-8') as f:
        data = json.load(f)
        rtn = data["change"]
        rtn = float(rtn) - 1
        # print(rtn)
    # print()
    
    if pd.isna(rtn):
        rtn = 0

    if sig > 0:
        if rtn > 0:
            tp += 1
            gain.append(rtn)
            if rtn > max_gain:
                max_gain = rtn

        else:
            fp += 1
            loss.append(rtn)
            if rtn < max_loss:
                max_loss = rtn


    elif sig == -1: 
        if rtn < 0:
            tn += 1
            gain.append(-rtn)
            if -rtn > max_gain:
                max_gain = -rtn

        else:
            fn += 1
            loss.append(-rtn)
            if -rtn < max_loss:
                max_loss = -rtn


print(max_gain, max_loss, tp+fp+tn+fn)
print(tp, fp, tn, fn)

accuracy = (tp + tn) / (tp + fp + tn + fn)
print(f'Accuracy: {accuracy}')
print(f'Average gain: {sum(gain) / len(gain)}')
print(f'Average loss: {sum(loss) / len(loss)}')
print(f'expected value: {(sum(gain) / len(gain))*accuracy + (sum(loss) / len(loss))*(1-accuracy)}')

930
0.009146877748460902 -0.007664946645959625 310
61 60 108 81
Accuracy: 0.5451612903225806
Average gain: 0.0015054186211874046
Average loss: -0.0014449745429685988
expected value: 0.0001634656013616095
